In [1]:
import pandas as pd
import re
import unicodedata
import glob
import os
from config_schema import SURVEY_SCHEMA, CNAE_MAP, QUEST_MAPPING, KEYWORD_RULES, SCORING_MAPS, POSTAL_CODE, SECTION_MAPPING, REGION_COORDINATES

def read_parse_csv(file_path: str) -> pd.DataFrame:
    """
    CSV reader with multiple encoding support.
    """
    encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'utf-16']
    
    for encoding in encodings:
        try:
            return pd.read_csv(file_path, encoding=encoding, sep=';')
        except UnicodeDecodeError:
            continue
        except Exception:
            continue
    
    print(f"WARNING: Could not read file {file_path}")
    return pd.DataFrame()

def normalize_questions_id(text: str) -> str:
    """
    Standardizes question string to create a unique key.
    """
    if pd.isna(text) or re.match(r'^\d', str(text)):
        return None
    
    text = str(text).lower().strip()
    text = re.sub(r'\s*\(.*?\)', '', text)
    text = unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode('ascii')
    text = text.replace('/', ' ').replace('-', ' ').replace('.', ' ')
    text = re.sub(r'"', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', '_', text)
    return text[:50]

def normalize_free_text(text: str, rule_type: str) -> str:
    """
    Basing on question value type access to keywords dictionary and categorize them.
    """
    if not text or str(text).lower() == 'nan':
        return "No especificado"
        
    clean_text = str(text).lower().strip()
    rules = KEYWORD_RULES.get(rule_type, {})

    for category, keywords in rules.items():
        for keyword in keywords:
            if keyword in clean_text:
                return category
                
    return str(text).title()

def normalize_response_value(row: pd.Series) -> str | int | float:
    """
    Normalize raw CSV answers into clean Data.
    """
    raw_val = str(row['Valor']).strip()
    field_id = str(row.get('internal_id', ''))
    
    schema_options = SURVEY_SCHEMA.get(field_id, [])
    if schema_options == ['NUMERIC']:
        # If number of employees is 0, we will consider it as 1 to avoid issues with KPIs calculations
        if "number_of_employees" in field_id and raw_val == "0":
            return 1

        digits = re.sub(r'[^\d\.,]', '', raw_val).replace(',', '.')
        try:
            # If it has decimal, return float else int
            return float(digits) if '.' in digits else int(digits)
        except ValueError:
            return 0

    if "cnae" in field_id or "profile" in field_id:
        if raw_val.isdigit() or re.match(r'^-?\d+\.?\d*$', raw_val):
            return CNAE_MAP.get(raw_val, "Otro")
        return normalize_free_text(raw_val, rule_type="sector")

    if field_id in ["erp_in_use", "crm_in_use", "powerbi_usage"]:
        # Check if value is in predefined csv questions options if not use keyword dictionary
        if raw_val in schema_options: 
            return raw_val
        return normalize_free_text(raw_val, rule_type="software")

    if "channel" in field_id or "comunicacion" in field_id:
        return normalize_free_text(raw_val, rule_type="channel")

    if "antivirus" in field_id:
        return normalize_free_text(raw_val, rule_type="antivirus")

    clean_val = raw_val.replace('"', '') 
    clean_val = re.sub(r'([Mm]enos (de|del)|[Mm]enor que)\s+', '<', clean_val, flags=re.IGNORECASE)
    clean_val = re.sub(r'([Mm]ás (de|del)|[Mm]ayor que)\s+', '>', clean_val, flags=re.IGNORECASE)
    clean_val = re.sub(r'Entre\s+(.*?)\s+y\s+(.*)', r'\1-\2', clean_val, flags=re.IGNORECASE)
    clean_val = re.sub(r'\s*-\s*', '-', clean_val)

    options = schema_options
    if not options or options == ["TEXT"]:
        return clean_val.title()

    for opt in options:
        if clean_val.lower() == opt.lower():
            return opt 
        opt_clean = opt.split('(')[0].strip()
        if len(opt_clean) > 3 and opt_clean.lower() in clean_val.lower():
            return opt
            
    return clean_val


def process_surveys(data_folder: str) -> pd.DataFrame:
    """
    Loops through CSVs, merges with schema, cleans data and unifies client data.
    """
    all_data = []
    files = glob.glob(os.path.join(data_folder, '*.csv'))
    print(f"Found {len(files)} files in {data_folder}")

    for file_path in files:            
        try:
            answers_df = read_parse_csv(file_path)
            if answers_df.empty or 'Campo' not in answers_df.columns:
                continue

            answers_df['questions'] = answers_df['Campo'].apply(normalize_questions_id)
            answers_df = answers_df.dropna(subset=['questions'])
            answers_df['internal_id'] = answers_df['questions'].map(QUEST_MAPPING).fillna(answers_df['questions'])
            answers_df['normalized_value'] = answers_df.apply(normalize_response_value, axis=1)

            # Create a single row dataframe for this client
            client_row = answers_df[['internal_id', 'normalized_value']].set_index('internal_id').T
            
            # Add metadata
            client_row.insert(0, 'source_file', os.path.basename(file_path))
            
            all_data.append(client_row)

        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")

    if all_data:
        return pd.concat(all_data, ignore_index=True)
    return pd.DataFrame()


DATA_DIR = './data/'
SCHEMA_FILE = './data/questions.csv'

print("Processing Survey Files...")
master_df = process_surveys(DATA_DIR)

Processing Survey Files...
Found 70 files in ./data/


In [2]:
def calculate_maturity_kpis(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates Digital Maturity Scores (0-100) 
    """
    
    def get_score(col_name, map_type):
        """
        Retrieves score using SCORING_MAPS, score rules predefined in config_schema.
        """
        if col_name not in df.columns:
            return 0
        
        if map_type == 'powerbi':
            # Case special for powerbi usage question, if it contains "ninguno" assign 0 else the company used powerbi
            return df[col_name].apply(lambda x: 0 if str(x).lower() == 'ninguno' else 100)
        
        if map_type == 'free_percentage':
            # For free percentage questions, we will consider the value as percentage if it's between 0 and 100, else 0
            def percentage_score(x):
                try:
                    value = float(x)
                    if 0 <= value <= 25:
                        return 25
                    elif 25 < value <= 50:
                        return 50
                    elif 50 < value <= 75:
                        return 75
                    elif 75 < value <= 100:
                        return 100
                except ValueError:
                    pass
                return 0
            
            return df[col_name].apply(percentage_score)

        # Select scoring maps from config schema
        mapping_dict = SCORING_MAPS.get(map_type, {})
        return df[col_name].map(mapping_dict).fillna(0)

    # Weights: Processes (40%), Infrastructure (30%), Collaboration (15%), PowerBI (15%)
    df['KPI_OPERATIONS'] = (
        get_score('key_processes_digitized_pct', 'percentage') * 0.4 + 
        get_score('it_infrastructure_type', 'infrastructure') * 0.3 +
        get_score('collaboration_tools_usage', 'binary') * 0.15 +
        get_score('powerbi_usage', 'powerbi') * 0.15
    )

    # Weights: Revenue (40%), AI (20%), Web Presence (20%), Marketing (20%)
    df['KPI_BUSINESS'] = (
        get_score('digital_revenue', 'revenue') * 0.4 +
        get_score('ai_for_automation_usage', 'ai') * 0.2 +
        get_score('active_internet_presence', 'binary') * 0.2 +
        get_score('digital_marketing_use', 'binary') * 0.2
    )

    # 25% each one
    df['KPI_SECURITY'] = (
        get_score('two_factor_authentication', 'binary') * 0.25 +
        get_score('continuity_and_recovery_plans', 'binary') * 0.25 +
        get_score('phishing_simulations', 'binary') * 0.25 +
        get_score('data_protection_compliance', 'binary') * 0.25
    )
    
    # Weights: Advanced Skills (25%), Cybersecurity Training (25%), Remote Work Policy (25%), Antivirus Usage (25%)
    df['KPI_CULTURE'] = (
        get_score('advanced_digital_skills_pct', 'percentage') * 0.25 +
        get_score('cybersecurity_training', 'binary') * 0.25 + 
        get_score('remote_work_acceptable_use_policy', 'binary') * 0.25 +
        get_score('employees_using_antivirus_pct', 'free_percentage') * 0.25
    )

    # Calculate dmi global score
    df['GLOBAL_SCORE'] = (
        df['KPI_OPERATIONS'] * 0.30 +
        df['KPI_SECURITY'] * 0.30 +
        df['KPI_BUSINESS'] * 0.20 +
        df['KPI_CULTURE'] * 0.20
    ).round(1)

    # Assign maturity labels based on global score
    def assign_label(score):
        if score >= 80:
            return 'Líder Digital'
        if score >= 60: 
            return 'Avanzado'
        if score >= 40: 
            return 'En Desarrollo'
        return 'Principiante Digital'

    df['MATURITY_LABEL'] = df['GLOBAL_SCORE'].apply(assign_label)
    return df

master_df = calculate_maturity_kpis(master_df)

In [3]:
import numpy as np
import json
from collections import defaultdict

def generate_benchmark_reference(df: pd.DataFrame) -> dict:
    """
    Takes the master dataframe with dmi calculated and aggregates it into 
    a json dictionary for benchmarking.
    """

    # Establish company size categories
    conditions = [
        (df['number_of_employees'] <= 10),
        (df['number_of_employees'] > 10) & (df['number_of_employees'] <= 50),
        (df['number_of_employees'] > 50) & (df['number_of_employees'] <= 250),
        (df['number_of_employees'] > 250)
    ]
    choices = ['Micro', 'Pequeña', 'Mediana', 'Grande']
    df['company_size'] = np.select(conditions, choices, default='Desconocido')

    df['province'] = df['company_postcode'].astype(str).str[:2].map(POSTAL_CODE).fillna('Desconocido')

    reference_data = {}

    # Group by sector + size ("Industrial" + "Small")
    groups_combinations = [
        df.groupby(['company_profile_cnae', 'company_size']),
        df.groupby(['company_profile_cnae', 'province']),
        df.groupby(['company_size', 'province'])]

    print(f"Generating benchmarks for {len(groups_combinations)}...")

    for grouped in groups_combinations:
        reference_data.update(calculate_reference_by_group(grouped))

    return reference_data

def calculate_reference_by_group(grouped) -> dict:
    """
    Generic function to calculate reference data by any grouping columns.
    """
    score_cols = [
        'KPI_OPERATIONS',
        'KPI_SECURITY',
        'KPI_BUSINESS', 
        'KPI_CULTURE',
        'GLOBAL_SCORE',
    ]
    
    # We check the % of companies that have these implemented (yes/no)
    adoption_cols = [
        'two_factor_authentication',
        'continuity_and_recovery_plans',
        'microsoft_365_usage',
        'remote_work_acceptable_use_policy',
        'phishing_simulations',
        'active_internet_presence',
        'data_protection_compliance',
        'regular_patching_and_updates',
        'incident_response_plan',
        'digital_marketing_use',
        'accessible_digital_sales_channels',
        'continuous_digital_training',
        'ai_for_automation_usage'
    ]
    
    # We check the most popular tools in each sector(top 3)
    market_cols = [
        'erp_in_use', 
        'crm_in_use', 
        'it_infrastructure_type',
        'antivirus_used',
        'powerbi_usage',
        'priority_assessment_area',
        'average_employee_age',
        'it_outsourcing_level',
        'digital_revenue'     
    ]

    reference_data = {}
    outsourcing_map = {'Bajo': 1, 'Medio': 2, 'Alto': 3}

    for (x, y), group in grouped:
        # Unique ID (eg. "Industrial_Small")
        combination_id = f"{x}_{y}"
        column_names = grouped.grouper.names

        # Minimum 3 samples to consider valid benchmark
        if len(group) < 3: 
            continue
            
        stats = {
            "meta": {
                column_names[0]: x,
                column_names[1]: y,
                "sample_size": len(group)
            },
            "scores": {},
            "adoption_rates": {},
            "market_leaders": {},
            "averages": {},
            "all_questions": defaultdict(dict)
        }
        
        for col in group.columns:
            if col in score_cols:
                score_stats = calculate_score_stats(group[col])
                stats["scores"][col] = score_stats
                
            elif col in adoption_cols:
                adoptation_rate_stats = calculate_adoption_rate(group[col], keyword='Sí|En producción')
                stats["adoption_rates"][col] = adoptation_rate_stats
                stats["all_questions"][SECTION_MAPPING[col]][col] = adoptation_rate_stats

            elif col in market_cols:
                stats["market_leaders"][col] = format_top_items(group[col])
                stats["all_questions"][SECTION_MAPPING[col]][col] = format_top_items(group[col])

            elif SURVEY_SCHEMA.get(col) == ['NUMERIC']:
                avg_val = group[col].mean()
                stats["averages"][col] = float(round(avg_val, 2))
                stats["all_questions"][SECTION_MAPPING[col]][col] = float(round(avg_val, 2))

            elif SURVEY_SCHEMA.get(col) == ['TEXT']:
                stats["all_questions"][SECTION_MAPPING[col]][col] = format_top_items(group[col])

            elif SURVEY_SCHEMA.get(col):
                stats["all_questions"][SECTION_MAPPING[col]][col] = format_top_items(group[col])

        if 'it_outsourcing_level' in group.columns:
            numeric_vals = group['it_outsourcing_level'].map(outsourcing_map).dropna()
            if not numeric_vals.empty:
                avg_val = numeric_vals.mean()
                # Convert back to text for display (1=Bajo, 2=Medio, 3=Alto)
                label = "Bajo" if avg_val < 1.5 else "Alto" if avg_val > 2.5 else "Medio"
                stats["averages"]['it_outsourcing_level'] = {
                    "avg_score": float(round(avg_val, 2)),
                    "label": label
                }    

        reference_data[combination_id] = stats
    return reference_data

def calculate_score_stats(series):
    """Calculate p25, median, p75 for a numeric series."""
    return {
        "p25": float(round(series.quantile(0.25), 1)),
        "median": float(round(series.median(), 1)),
        "p75": float(round(series.quantile(0.75), 1))
    }

def calculate_adoption_rate(series, keyword):
    """Calculate percentage of rows containing keyword."""
    is_match = series.astype(str).str.contains(keyword, case=False, regex=True)
    return float(round(is_match.mean() * 100, 1))

def format_top_items(series, n=3):
    """Format top N items by frequency as list of dicts."""
    counts = series.value_counts(normalize=True).head(n)
    return [
        {"tool": name, "share_pct": float(round(share * 100, 1))}
        for name, share in counts.items()
    ]


benchmark_json = generate_benchmark_reference(master_df)

with open('benchmark_data.json', 'w', encoding='utf-8') as f:
    json.dump(benchmark_json, f, indent=4, ensure_ascii=False)
    print("Success! 'benchmark_data.json' has been created.")

Generating benchmarks for 3...
Success! 'benchmark_data.json' has been created.


/var/folders/7p/_fsqkrtn0378mkm7gx5z55lc0000gn/T/ipykernel_83657/3859454213.py:86: FutureWarning: DataFrameGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  column_names = grouped.grouper.names
/var/folders/7p/_fsqkrtn0378mkm7gx5z55lc0000gn/T/ipykernel_83657/3859454213.py:86: FutureWarning: DataFrameGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  column_names = grouped.grouper.names
/var/folders/7p/_fsqkrtn0378mkm7gx5z55lc0000gn/T/ipykernel_83657/3859454213.py:86: FutureWarning: DataFrameGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  column_names = grouped.grouper.names
/var/folders/7p/_fsqkrtn0378mkm7gx5z55lc0000gn/T/ipykernel_83657/3859454213.py:86: FutureWarning: DataFrameGroupBy.grouper is deprecated and will be removed in a future version of pandas.
  column_names = grouped.grouper.names
/var/folders/7p/_fsqkrtn0378mkm7gx5z55lc0000gn/T/ipykernel_83657/3859454213.py:86: FutureWar

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import base64
from io import BytesIO
from jinja2 import Template
import datetime
import plotly.express as px

# Set visual style for professional charts
sns.set_theme(style="whitegrid")

def plot_to_base64(fig):
    """Converts plot to base64 string for HTML embedding"""
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
    buf.seek(0)
    img_str = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig) 
    return img_str

    
def create_kpi_methodology_table():
    """Generates the HTML table explaining the KPI composition"""
    return """
    <table class="methodology-table">
        <tr>
            <th>KPI</th>
            <th>Qué mide</th>
            <th>Columnas / Preguntas utilizadas</th>
        </tr>
        <tr>
            <td><strong>Eficiencia Operativa</strong></td>
            <td>Nivel de modernización de la infraestructura y procesos.</td>
            <td>
                <ul>
                    <li><em>% Procesos digitalizados (40%)</em></li>
                    <li><em>Infraestructura (Cloud vs On-Prem) (30%)</em></li>
                    <li><em>Uso de herramientas colaborativas (15%)</em></li>
                    <li><em>¿Usas PowerBI para la analítica empresarias de tu negocio? (15%)</em></li>
                </ul>
            </td>
        </tr>
        <tr>
            <td><strong>Negocio Digital</strong></td>
            <td>Capacidad de generar valor e ingresos a través de la tecnología.</td>
            <td>
                <ul>
                    <li><em>% Ingresos Digitales (40%)</em></li>
                    <li><em>Uso de IA para automatización (20%)</em></li>
                    <li><em>Presencia activa en Internet (20%)</em></li>
                    <li><em>Uso de marketing digital (20%)</em></li>
                </ul>
            </td>
        </tr>
        <tr>
            <td><strong>Ciberseguridad</strong></td>
            <td>Nivel de protección y cumplimiento normativo (Higiene Digital).</td>
            <td>
                <ul>
                    <li><em>Autenticación de Doble Factor (2FA) (25%)</em></li>
                    <li><em>Planes de Continuidad/Backups (25%)</em></li>
                    <li><em>Simulacros de Phishing (25%)</em></li>
                    <li><em>Cumplimiento RGPD/Datos (25%)</em></li>
                </ul>
            </td>
        </tr>
        <tr>
            <td><strong>Cultura Digital</strong></td>
            <td>Capacitación y preparación del capital humano.</td>
            <td>
                <ul>
                    <li><em>% Empleados con skills avanzados (25%)</em></li>
                    <li><em>Formación específica en Ciberseguridad (25%)</em></li>
                    <li><em>¿Existe política de uso aceptable para el teletrabajo? (25%)</em></li>
                    <li><em>¿Qué porcentaje de tus empleados usan antivirus? (25%)</em></li>
                </ul>
            </td>
        </tr>
    </table>
    """

def create_scoring_logic_html():
    """
    Generates a clean HTML grid showing how text answers are converted to numbers.
    """
    scoring_maps = {
        "Respuestas Estándar": {
            'Sí': 100,
            'Parcial / En desarrollo': 50, 
            'No / Ninguno': 0
        },
        "Infraestructura TI": {
            'Cloud (Nube)': 100, 
            'Híbrida': 70, 
            'On-premise (Local)': 30
        },
        "Inteligencia Artificial": {
            'En producción': 100, 
            'En piloto': 75, 
            'Explorando': 40, 
            'No': 0
        },
        "Ingresos / Digitalización": {
            'Alto (>60% / 76-100%)': 100,
            'Medio (30-60% / 51-75%)': 75, 
            'Bajo (10-30% / 26-50%)': 50, 
            'Nulo (<10% / 0-25%)': 25
        }
    }

    # CSS for the grid
    html = """
    <style>
        .scoring-grid { display: flex; flex-wrap: wrap; gap: 15px; margin-top: 20px; }
        .scoring-card { flex: 1; min-width: 180px; background: #f8f9fa; border-left: 4px solid #0062a4; border-radius: 4px; padding: 10px; box-shadow: 0 1px 3px rgba(0,0,0,0.1); }
        .scoring-card h4 { color: #0062a4; margin: 0 0 10px 0; font-size: 13px; text-transform: uppercase; border-bottom: 1px solid #ddd; padding-bottom: 5px; }
        .score-row { display: flex; justify-content: space-between; font-size: 11px; margin-bottom: 4px; color: #555; }
        .score-val { font-weight: bold; color: #27ae60; }
    </style>
    <div class='scoring-grid'>
    """

    for category, items in scoring_maps.items():
        html += f"<div class='scoring-card'><h4>{category}</h4>"
        for label, score in items.items():
            html += f"<div class='score-row'><span>{label}</span><span class='score-val'>{score} pts</span></div>"
        html += "</div>"
    
    html += "</div>"
    return html


In [5]:
master_df.head(20).style

internal_id,source_file,company_profile_cnae,company_postcode,number_of_employees,average_employee_age,number_of_clients,number_of_suppliers,annual_revenue,it_outsourcing_level,remote_work_acceptable_use_policy,secure_remote_access,two_factor_authentication,it_infrastructure_type,key_processes_digitized_pct,erp_in_use,crm_in_use,ai_for_automation_usage,database_type,powerbi_usage,advanced_digital_skills_pct,microsoft_365_usage,collaboration_tools_usage,continuous_digital_training,cybersecurity_training,ftfe_training,phishing_simulations,active_internet_presence,active_social_media_management,digital_marketing_use,visitor_follower_analysis,accessible_digital_sales_channels,digital_revenue,usual_customer_communication_channel,preferred_customer_communication_channel,antivirus_used,employees_using_antivirus_pct,regular_patching_and_updates,network_controls_implemented,documented_account_lifecycle_process,clear_roles_and_privileges,incident_response_plan,continuity_and_recovery_plans,data_protection_compliance,legal_and_compliance_training,priority_assessment_area,KPI_OPERATIONS,KPI_BUSINESS,KPI_SECURITY,KPI_CULTURE,GLOBAL_SCORE,MATURITY_LABEL,company_size,province
0,mock_results_3525.csv,Otro,1036,87,30-40,2125,95,>20M,Medio,Sí,No,Sí,Híbrida,0-25%,Oracle,Ninguno,No,OnPremise,Otro,51-75%,Si,Parcial,Sí,Sí,Si,Sí,Sí,Ocasional,En evaluación,No,En Desarrollo,>60%,Email,Email,Microsoft Defender,52,Parcial,Sí,Parcial,Parcial,No,En desarrollo,Parcial,Parcial,Gestión de incidencias y continuidad de negocio,53.500000,70.000000,75.000000,87.500000,70.000000,Avanzado,Mediana,Cáceres
1,mock_results_2941.csv,Tecnología,1667,139,>50,1054,90,<1M,Bajo,Sí,Parcial,No,Híbrida,26-50%,Microsoft,Otro,En producción,OnPremise,Power BI,26-50%,Si,Parcial,Ocasional,<1 vez al año,No,Sí,No,Sí,En evaluación,Parcial,Sí,10-30%,Videollamada,WhatsApp,McAfee,42,No,Sí,No,Sí,Sí,No,No,Sí,Gestión de incidencias y continuidad de negocio,63.500000,50.000000,25.000000,50.000000,46.600000,En Desarrollo,Mediana,Cuenca
2,mock_results_3849.csv,Servicios,0635,108,30-40,4471,42,5-20M,Bajo,No,Parcial,No,Híbrida,51-75%,Oracle,Otro,En piloto,OnPremise,Tableau,76-100%,Si,Sí,No,No,Si,Sí,No,Sí,Sí,No,Sí,30-60%,Videollamada,Teléfono,McAfee,59,Parcial,No,No,No,Sí,En desarrollo,Sí,Sí,Protección de datos y propiedad intelectual,81.000000,65.000000,62.500000,43.750000,64.800000,Avanzado,Mediana,Badajoz
3,form_data_69666a746084f_cealvet.csv,Comercio,43500,7,30-40,51,16,1-5M,Bajo,Sí,Sí,Parcial,On-premise,76-100%,Facturascript,Facturascript,En piloto,OnPremise,Power BI,76-100%,Sí,Sí,Sí,<1 vez al año,Sí,No,Sí,Sí,No,Sí,No,<10%,Teléfono,Teléfono,Panda Security,100,Sí,Sí,No,Sí,No,Sí,Sí,No,Procesos y automatización,79.000000,45.000000,62.500000,75.000000,66.400000,Avanzado,Micro,Tarragona
4,form_data_693941bba67fc_siesystems.csv,Industrial,26006,12,30-40,25,5,<1M,Bajo,No,Sí,Sí,Híbrida,26-50%,A3 / Wolters Kluwer,Wolfcrm,Explorando,Cloud,Power BI,51-75%,Sí,Sí,Ocasional,<1 vez al año,Sí,Puntual,Sí,Sí,Sí,Sí,No,10-30%,WhatsApp,Email,Eset Nod32,100,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Sí,Infraestructuras y conectividad,71.000000,68.000000,75.000000,43.750000,66.200000,Avanzado,Pequeña,La Rioja
5,mock_results_3532.csv,Comercio,0500,70,>50,874,70,>20M,Bajo,No,Sí,No,Híbrida,76-100%,Otro,Otro,En producción,Cloud,Tableau,0-25%,No,Sí,Sí,No,No,Sí,Parcial,Ocasional,No,No,No,30-60%,Email,Email,McAfee,5,Parcial,No,Parcial,Sí,En elaboración,Sí,No,No,Procesos y automatización,91.000000,60.000000,50.000000,12.500000,56.800000,En Desarrollo,Mediana,Ávila
6,mock_results_4063.csv,Otro,3556,142,>50,856,14,>20M,Bajo,No,Parcial,No,Híbrida,26-50%,SAP,Ninguno,No,Cloud,Excel,26-50%,No,No,Ocasional,<1 vez al año,Si,Sí,Parcial,No,Sí,No,No,10-30%,WhatsApp,Email,Microsoft Defender,52,Parcial,Parcial,Sí,No,Sí,No,Sí,No,Gestión de identidades y control de accesos,56.000000,50.000000,50.000000,31.250000,48.000000,En Desarrollo,Mediana,Palmas (Las)
7,mock_results_5976.csv,Comercio,3929,174,30-40,775,30,1-5M,Medio,Sí,Sí,No,Híbrida,0-25%,Ninguno,Zoho,E

In [6]:
INTERACTIVE_REPORT_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Informe de Mercado</title>
    <style>
        body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; padding: 0; color: #333; background: #f4f7f6; }
        .container { width: 900px; margin: 0 auto; background: white; padding: 40px; box-shadow: 0 0 10px rgba(0,0,0,0.1); }
        header { border-bottom: 3px solid #0062a4; padding-bottom: 20px; margin-bottom: 30px; }
        h1 { color: #0062a4; font-size: 28px; margin: 0; }
        h2 { color: #0062a4; border-left: 5px solid #2ecc71; padding-left: 15px; margin-bottom: 20px; page-break-after: avoid; }
        
        .logo { height: 100px; width: 300px; padding-left: 2rem; }
        .header { background: #0062a4; width: auto; height: 150px; align-content: center; }
        
        .methodology-box { background: #fff; border: 1px solid #ddd; padding: 15px; border-radius: 5px; margin-bottom: 40px; }
        .methodology-table { width: 100%; border-collapse: collapse; font-size: 13px; }
        .methodology-table th { background: #0062a4; color: white; padding: 8px; text-align: left; }
        .methodology-table td { border-bottom: 1px solid #eee; padding: 8px; vertical-align: top; }
        
        .kpi-row { display: flex; justify-content: space-between; margin-bottom: 40px; }
        .kpi-card { background: #eef2f5; padding: 20px; border-radius: 8px; width: 30%; text-align: center; border-top: 4px solid #0062a4; }
        .kpi-value { font-size: 32px; font-weight: bold; color: #0062a4; }
        
        .section { margin-bottom: 50px; page-break-inside: avoid; }
        .chart-box { text-align: center; margin: 20px 0; border: 1px solid #eee; padding: 10px; border-radius: 5px; }
        img { max-width: 100%; height: auto; }
        .footer { text-align: center; font-size: 11px; color: #999; margin-top: 50px; border-top: 1px solid #eee; padding-top: 10px; }
    </style>
</head>
<body>

<div class="container">
    <header class="header">
        <img src="https://disi.tareando.es/img/disiLogoTareandoBlanco.png?v=" alt="Tareando Logo" class="logo"/>
    </header>
    <header>
        <h1>Análisis Digital del Mercado</h1>
        <div class="subtitle">Informe Estratégico | {{ date }}</div>
    </header>

    <div class="section">
        <h2>Descripción breve y objetivo del informe</h2>
        <p>Este informe presenta un análisis exhaustivo del estado de madurez digital de las empresas encuestadas, identificando fortalezas, debilidades y oportunidades clave para la transformación digital. El objetivo es proporcionar insights accionables para que puedan diseñar estrategias efectivas de digitalización.</p>
    </div>

    <div class="section">
    <h2>Indicadores Clave</h2>
    <p>
        Antes de profundizar, presentamos los indicadores generales del mercado. Estos tres números resumen la salud digital de las {{ total_companies }} empresas analizadas, destacando el volumen de la muestra, la calidad media y, crucialmente, el nivel de exposición al riesgo.
    </p>

    <div class="kpi-row">
        <div class="kpi-card">
            <span class="kpi-value">{{ total_companies }}</span><br>
            Empresas Analizadas<br>
        </div>

        <div class="kpi-card">
            <span class="kpi-value">{{ avg_score }}</span><br>
            Índice de Madurez (0-100)<br>
            <small style="color: #666; font-size: 11px;">Promedio de mercado</small>
        </div>

        <div class="kpi-card" style="border-top: 4px solid #FFA500;"> <span class="kpi-value" style="color:#FFA500;">{{ risk_pct }}%</span><br>
            Tasa de Vulnerabilidad<br>
            <small style="color: #666; font-size: 11px;">Empresas con seguridad crítica (<40/100)</small>
        </div>        
    </div>

    <div style="background-color: #fff8e1; padding: 15px; border-radius: 5px; border-left: 5px solid #FFA500; font-size: 13px; margin-top: 20px;">
        <strong>ℹ️ ¿Qué significa la Tasa de Vulnerabilidad?</strong><br>
        Este indicador revela que el <strong>{{ risk_pct }}% del mercado opera en "Zona de Peligro"</strong>. 
        Estas empresas han obtenido menos de 40 puntos en Ciberseguridad, lo que implica que carecen de los "cortafuegos" básicos (como copias de seguridad verificadas o autenticación robusta). 
        No es solo un área de mejora; es una estadística de <strong>probabilidad de supervivencia</strong> ante un ciberataque.
    </div>
</div>

    <div class="section">
        <h2>Lógica de Puntuación</h2>
        <p>La puntuación global de madurez digital se calcula a partir de cuatro dimensiones clave: Operaciones, Negocio Digital, Ciberseguridad y Cultura Digital. Cada dimensión se compone de varias preguntas específicas que se ponderan para reflejar su importancia relativa en la madurez digital general.</p>
        <div class="methodology-box">{{ scoring_logic }}</div>
    </div>

    <div class="section">
        <h2>Metodología de evaluación</h2>
        <p>
            Para obtener una radiografía precisa de la madurez digital, no nos basamos en una única métrica. Nuestro algoritmo evalúa la empresa desde cuatro dimensiones estratégicas (KPIs), asignando un peso específico a cada respuesta según su impacto real en el negocio.
        </p>
        <p>
            La puntuación global (0-100) es el resultado ponderado de estos cuatro pilares. A continuación, desglosamos la "fórmula" exacta utilizada para calcular cada indicador, detallando qué preguntas componen cada bloque y cuánto influyen en la nota final:
        </p>
        <div class="methodology-box">{{ methodology_table }}</div>
    </div>

    <div class="section">
        <h2>Comparativa promedio participantes vs líderes</h2>
        <p>Este gráfico de radar compara el perfil de madurez digital promedio del mercado con el de los líderes (empresas en el percentil 75). Permite identificar áreas donde el mercado en general está rezagado y dónde se encuentran las mejores prácticas.</p>
        <div class="chart-box">
        {{ chart_radar_url }}
        </div>
    </div>

    <div class="section">
        <h2>Distribución de Madurez del Mercado</h2>
        <p>Este histograma muestra cómo se distribuyen las puntuaciones globales de madurez digital entre las empresas encuestadas, destacando la media y la variabilidad del mercado.</p>
        <ul>
            <li><strong>Micro:</strong> <10 empleados</li>
            <li><strong>Pequeña:</strong> 11-50 empleados</li>
            <li><strong>Mediana:</strong> 51-250 empleados</li>
            <li><strong>Grande:</strong> >250 empleados</li>
        </ul>
        <div class="chart-box">
        {{ chart_distplot_url }}
        </div>
    </div>

    <div class="section">
        <h2>Desempeño por Sector</h2>
        <p>
            Análisis comparativo de la madurez digital agrupado por sectores de actividad. 
            El gráfico de cajas (Boxplot) permite identificar qué industrias lideran la transformación 
            y cuáles presentan una mayor dispersión interna (brecha entre líderes y rezagados dentro del mismo sector).
        </p>
        <div class="chart-box">
        {{ chart_boxplot_url }}
        </div>
    </div>

    <div class="section">
        <h2>Análisis Regional</h2>
        <p>
            Mapa de calor que visualiza la intensidad digital por ubicación geográfica. 
            Este indicador ayuda a detectar polos de innovación tecnológica y correlaciones 
            entre la ubicación de la sede y el nivel de adopción de herramientas digitales.
            Cabe destacar que se ha tomado los códigos postales y se han agrupado por provincias, asignando a cada una un color basado en el promedio de madurez digital de las empresas ubicadas en esa provincia.
        </p>
        <div class="chart-box">
        {{ chart_region_url }}
        </div>
    </div>


    <div class="section">
        <h2>Prioridades Estratégicas</h2>
        <p>
            ¿Dónde ponen el foco las empresas hoy? Este gráfico jerarquiza las áreas que 
            las compañías han marcado como "críticas" para su mejora (ej. Ciberseguridad, Ventas, Eficiencia). 
            Contrasta la intención estratégica con la realidad operativa observada.
        </p>
        <div class="chart-box">
        {{ chart_priority_url }}
        </div>
    </div>

    <div class="section">
        <h2>Uso del ERP, distribución erp y tecnología</h2>
        <p>El ERP es la columna vertebral de la digitalización. A continuación analizamos tres dimensiones:</p>
        <ul>
            <li><strong>Impacto:</strong> Cuánto mejora la calificación global al tener ERP.</li>
            <li><strong>Adopción:</strong> Porcentaje de implantación de cada tecnología.</li>
            <li><strong>Distribución por Tamaño:</strong> Porcentaje de adopción por tamaño de empresa.</li>
        </ul>
        <div style="display: flex; justify-content: space-between;">
            <div style="width: 48%;" class="chart-box">{{ chart_driver_erp_url }}</div>
            <div style="width: 48%;" class="chart-box">{{ chart_pie_erp_url }}</div>
        </div>
        <div class="chart-box">
        {{ chart_erp_distplot_url}}
        </div>
    </div>

    <div class="section">
        <h2>Uso del CRM, distribución crm y tecnología</h2>
        <p>El CRM es una herramienta clave para la gestión de relaciones con clientes. A continuación analizamos tres dimensiones:</p>
        <ul>
            <li><strong>Impacto:</strong> Cuánto mejora la calificación global al tener CRM.</li>
            <li><strong>Adopción:</strong> Porcentaje de implantación de cada tecnología.</li>
            <li><strong>Distribución por Tamaño:</strong> Porcentaje de adopción por tamaño de empresa.</li>
        </ul>
        <div style="display: flex; justify-content: space-between;">
            <div style="width: 48%;" class="chart-box">{{ chart_driver_crm_url }}</div>
            <div style="width: 48%;" class="chart-box">{{ chart_pie_crm_url }}</div>
        </div>
        <div class="chart-box">
        {{ chart_crm_distplot_url}}
        </div>
    </div>

    <div class="section">
        <h2>Uso de ERP y CRM en el mercado</h2>
        <p>
            Desglose de las herramientas tecnológicas específicas que dominan el mercado. 
        </p>
        <div style="display: flex; justify-content: space-between;">
            <div style="width: 48%;" class="chart-box">{{ chart_usage_crm_url }}</div>
            <div style="width: 48%;" class="chart-box">{{ chart_usage_erp_url }}</div>
        </div>
    </div>

    <div class="section">
        <h2>Adopción en ciberseguridad e Infraestructura</h2>
        <p>
            Evaluación crítica del riesgo tecnológico. A la izquierda, el cumplimiento de controles básicos de seguridad 
            (MFA, Backups, Planes de contingencia). A la derecha, cómo la modernización de la infraestructura (Cloud vs Híbrida vs On-Premise) 
            actúa como palanca aceleradora de la madurez digital.
        </p>
        <div style="display: flex; justify-content: space-between;">
            <div style="width: 48%;" class="chart-box">{{ chart_riskbar_url }}</div>
            <div style="width: 48%;" class="chart-box">{{ chart_driver_cloud_url }}</div>
        </div>
    </div>


    <div class="section">
        <h2>Madurez Digital vs Ingresos por empleado</h2>
        <p>
            Este análisis cruza la Madurez Digital con la Eficiencia Económica (Ingresos/Empleado). 
            El tamaño y color de las burbujas indican el tamaño de la plantilla, permitiendo detectar 
            <strong>"Micro-Gigantes"</strong>: empresas pequeñas altamente digitalizadas y eficientes.
        </p>
        <div class="chart-box">
        {{ chart_bubble_url }}
        </div>
    </div>

    <div class="section">
        <h2>Factor Humano y Legado Tecnológico</h2>
        <p>
            Análisis de correlación entre la media de edad de la plantilla y el tipo de infraestructura (Cloud vs Híbrida vs On-Premise). 
            Busca identificar si existe una "Trampa del Legado", donde estructuras más tradicionales muestran resistencia 
            al cambio hacia tecnologías en la nube.
        </p>
        <div class="chart-box">
        {{ chart_age_cloud_url }}
        </div>
    </div>

    <div class="section">
        <h2>Tracción Digital (Marketing vs Retorno)</h2>
        <p>
            Verificación de la efectividad de la inversión. Comparamos a las empresas que invierten en Marketing Digital 
            frente a sus ingresos digitales reales. El objetivo es detectar gasto sin retorno 
            y confirmar si la inversión publicitaria se traduce en ventas tangibles.
        </p>
        <div class="chart-box">
        {{ chart_digital_traction_url }}
        </div>
    </div>

    <div class="section" style="page-break-inside: avoid;">
        <h2>Conclusión</h2>
        <p>Basándonos en la correlación entre los indicadores (KPIs) y las prioridades declaradas por los participantes, presentamos el diagnóstico final del ecosistema:</p>
        {{ dynamic_conclusion_html }}
        
    </div>
    
    <div class="footer">Informe generado automáticamente | Valores agregados mediante empresas encuestadas</div>
</div>
</body>
</html>
"""

In [7]:
import plotly.graph_objects as go
from plotly.colors import n_colors

def create_int_radar_chart(df):
    categories = ['Eficiencia Operativa', 'Ciberseguridad', 'Negocio Digital', 'Cultura Digital']
    kpi_cols = ['KPI_OPERATIONS', 'KPI_SECURITY', 'KPI_BUSINESS', 'KPI_CULTURE']

    values_avg = [df[col].mean() for col in kpi_cols]
    values_leaders = [df[col].quantile(0.75) for col in kpi_cols]
    fig = go.Figure()

    fig.add_trace(go.Scatterpolar(
        r=values_avg,
        theta=categories,
        fill='toself',
        name='Promedio',
        marker=dict(color='orange')
    ))
    fig.add_trace(go.Scatterpolar(
        r=values_leaders,
        theta=categories,
        fill='toself',
        name='Líderes',
        marker=dict(color='purple')
    ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(
            visible=True,
            range=[0, 100],
            )),
        showlegend=True
    )

    return fig.to_html(full_html=False, include_plotlyjs='cdn')

def create_distplot_chart(df):
    all_groups = df.groupby('company_size')['GLOBAL_SCORE']
    group_labels = ['Micro', 'Pequeña', 'Mediana', 'Grande']
    existing_labels = []
    hist_data = []

    for label in group_labels:
        if label in all_groups.groups:
            data = all_groups.get_group(label)
            if len(data) > 1:
                hist_data.append(data.to_list())
                existing_labels.append(label)

    colors = n_colors('rgb(5, 200, 200)', 'rgb(200, 10, 10)', len(existing_labels), colortype='rgb')

    if not hist_data:
        return "<p>No hay datos suficientes para generar el gráfico de distribución.</p>"
    
    fig = go.Figure()
    for data, color, label in zip(hist_data, colors, existing_labels):
        fig.add_trace(go.Violin(x=data, name=label, line_color=color, box_visible=True, meanline_visible=True))

    fig.update_traces(orientation='h', side='positive', width=2, points=False)
    fig.update_layout(
        xaxis_showgrid=False, 
        xaxis_zeroline=False,
        xaxis_title='Puntuación Global de Madurez Digital',
        yaxis_title='Tamaño de Empresa',
    )

    return fig.to_html(full_html=False, include_plotlyjs='cdn')

def create_boxplot_chart(df):
    order = df.groupby('company_profile_cnae')['GLOBAL_SCORE'].median().sort_values(ascending=False).index
    fig = px.box(df, x='company_profile_cnae', y='GLOBAL_SCORE', color='company_profile_cnae', category_orders={'company_profile_cnae': order},
                 labels={'company_profile_cnae': 'Sector', 'GLOBAL_SCORE': 'Puntuación Global'})
    return fig.to_html(full_html=False, include_plotlyjs='cdn')

def create_riskbar_chart(df):
    risk_cols = ['two_factor_authentication', 'continuity_and_recovery_plans', 'regular_patching_and_updates', 'phishing_simulations']
    risk_labels = ['2FA', 'Backups/BCP', 'Parcheo', 'Phishing Tests']
    adoption = {}
    for col in risk_cols:
        if col in df.columns:
            adoption[col] = round(df[col].astype(str).str.contains(r'sí', case=False).mean() * 100)

    fig = go.Figure(
        data=[
            go.Bar(
                y=list(adoption.values()),
                x=risk_labels,
            )
        ],
        layout=dict(
            barcornerradius=10,
        )
    )

    fig.update_traces(marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)',
                  marker_line_width=1.5, opacity=0.6)

    fig.update_layout(
        title='Adopción de Ciberseguridad (%)',
        xaxis=dict(title='Controles de Seguridad'),
        yaxis=dict(title='Porcentaje de Adopción')
    )

    return fig.to_html(full_html=False, include_plotlyjs='cdn')


def create_int_priority_chart(df):
    if 'priority_assessment_area' not in df.columns: 
        return "<p>No hay datos suficientes para generar el gráfico de prioridades.</p>"

    priority_counts = df['priority_assessment_area'].value_counts()
    fig = go.Figure(
        data=[
            go.Bar(
                y=priority_counts.index,
                x=priority_counts.values,
                orientation='h',
            )
        ],
    )

    fig.update_traces(marker_color='rgb(158,202,225)', marker_line_color='rgb(8,48,107)',
                  marker_line_width=1.5, opacity=0.6)

    fig.update_layout(
        title='Áreas de Evaluación Prioritarias',
        xaxis=dict(title='Número de Empresas'),
        yaxis=dict(title='Área de Evaluación')
    )

    return fig.to_html(full_html=False, include_plotlyjs='cdn')


def create_pie_chart(df):
    clean_series = df['erp_in_use'].fillna("No especificado")
    counts = clean_series.value_counts().head(6) 

    fig_erp = go.Figure(
        data=[
            go.Pie(
                labels=counts.index,
                values=counts.values,
            )
        ]
    )

    fig_erp.update_traces(marker=dict(colors=sns.color_palette("pastel").as_hex()))

    fig_erp.update_layout(
        title='Cuota de Mercado ERP'
    )

    clean_series_crm = df['crm_in_use'].fillna("No especificado")
    counts_crm = clean_series_crm.value_counts().head(6)
    fig_crm = go.Figure(
        data=[
            go.Pie(
                labels=counts_crm.index,
                values=counts_crm.values,
            )
        ]
    )
    fig_crm.update_traces(marker=dict(colors=sns.color_palette("pastel").as_hex()))
    fig_crm.update_layout(
        title='Cuota de Mercado CRM'
    )

    return fig_erp.to_html(full_html=False, include_plotlyjs='cdn'), fig_crm.to_html(full_html=False, include_plotlyjs='cdn')


def create_int_driver_charts(df):
    """
    Generates charts showing what drives high maturity (Correlation Analysis).
    """
    # 1. CRM Impact
    df['has_crm'] = df['crm_in_use'].apply(lambda x: 'Sin CRM' if str(x).lower() in ['ninguno'] else 'Con CRM')
    
    fig_crm = go.Figure()
    for label in df['has_crm'].unique():
        fig_crm.add_trace(go.Box(
            y=df[df['has_crm'] == label]['GLOBAL_SCORE'],
            name=label
        ))

    fig_crm.update_layout(
        title='Impacto del CRM en la Madurez Global',
        yaxis=dict(title='Puntuación Global'),
        xaxis=dict(title='Uso de CRM')
    )
    
    # 2. Infrastructure Impact
    order = ['On-premise', 'Híbrida', 'Cloud']
    existing_order = [x for x in order if x in df['it_infrastructure_type'].unique()]
    
    fig_cloud = go.Figure()
    for label in existing_order:
        fig_cloud.add_trace(go.Box(
            y=df[df['it_infrastructure_type'] == label]['GLOBAL_SCORE'],
            name=label
        ))

    fig_cloud.update_layout(
        title='Impacto de la Nube en la Madurez Global',
        yaxis=dict(title='Puntuación Global'),
        xaxis=dict(title='Tipo de Infraestructura TI')
    )

    # 3. ERP Impact
    df['has_erp'] = df['erp_in_use'].apply(lambda x: 'Sin ERP' if str(x).lower() in ['ninguno'] else 'Con ERP')

    fig_erp = go.Figure()
    for label in df['has_erp'].unique():
        fig_erp.add_trace(go.Box(
            y=df[df['has_erp'] == label]['GLOBAL_SCORE'],
            name=label
        ))

    fig_erp.update_layout(
        title='Impacto del ERP en la Madurez Global',
        yaxis=dict(title='Puntuación Global'),
        xaxis=dict(title='Uso de ERP')
    )
    
    return fig_erp.to_html(full_html=False, include_plotlyjs='cdn'), fig_crm.to_html(full_html=False, include_plotlyjs='cdn'), fig_cloud.to_html(full_html=False, include_plotlyjs='cdn')


def create_erp_crm_distplot_chart(df):
    df = df.copy()
    df['has_erp'] = df['erp_in_use'].apply(lambda x: 'Sin ERP' if str(x).lower() in ['ninguno'] else 'Con ERP')
    grouped = df.groupby(['company_size', 'has_erp']).size().reset_index(name='count')    
    grouped['percentage'] = grouped.groupby('company_size')['count'].transform(lambda x: x / x.sum() * 100)
    
    size_order = ['Micro', 'Pequeña', 'Mediana', 'Grande']

    fig_erp = px.bar(
        grouped,
        x='percentage',
        y='company_size',
        color='has_erp',
        orientation='h',
        text_auto='.1f',
        category_orders={'company_size': size_order},
        title="Distribución de ERP: Adopción por Tamaño de Empresa",
        labels={'percentage': 'Porcentaje de Adopción (%)', 'company_size': 'Tamaño de empresa', 'has_erp': 'ERP'},
        color_discrete_map={
            'Con ERP': '#8DE5A1',
            'Sin ERP': '#CFCFCF'
        }
    )
    fig_erp.update_layout(legend_title_text="")

    df['has_crm'] = df['crm_in_use'].apply(lambda x: 'Sin CRM' if str(x).lower() in ['ninguno'] else 'Con CRM')
    grouped_crm = df.groupby(['company_size', 'has_crm']).size().reset_index(name='count')    
    grouped_crm['percentage'] = grouped_crm.groupby('company_size')['count'].transform(lambda x: x / x.sum() * 100)
    fig_crm = px.bar(
        grouped_crm,
        x='percentage',
        y='company_size',
        color='has_crm',
        orientation='h',
        text_auto='.1f',
        category_orders={'company_size': size_order},
        title="Distribución de CRM: Adopción por Tamaño de Empresa",
        labels={'percentage': 'Porcentaje de Adopción (%)', 'company_size': 'Tamaño de empresa', 'has_crm': 'CRM'},
        color_discrete_map={
            'Con CRM': '#8DE5A1',
            'Sin CRM': '#CFCFCF'
        }
    )
    fig_crm.update_layout(legend_title_text="")
    
    return fig_erp.to_html(full_html=False, include_plotlyjs='cdn'), fig_crm.to_html(full_html=False, include_plotlyjs='cdn')


def create_int_opportunies_charts(df):
    no_crm_mask = df['crm_in_use'].astype(str).str.contains('ninguno', case=False)
    no_crm_count = no_crm_mask.sum()
    total_count = len(df)
    has_crm_count = total_count - no_crm_count
    
    sizes = [no_crm_count, has_crm_count]
    labels = [f'Sin CRM ({no_crm_count})', f'Con CRM ({has_crm_count})']
    colors = ['#CFCFCF', '#8DE5A1'] 
    
    fig = go.Figure(
        data=[
            go.Pie(
                labels=labels,
                values=sizes,
                marker=dict(colors=colors),
                pull=[0, 0.1])
        ]
    )

    fig.update_layout(
        title='Mercado CRM'
    )

    no_erp_mask = df['erp_in_use'].astype(str).str.contains('ninguno', case=False)
    no_erp_count = no_erp_mask.sum()
    has_erp_count = total_count - no_erp_count
    
    sizes_erp = [no_erp_count, has_erp_count]
    labels_erp = [f'Sin ERP ({no_erp_count})', f'Con ERP ({has_erp_count})']
    colors_erp = ['#CFCFCF', '#8DE5A1']
    fig_erp = go.Figure(
        data=[
            go.Pie(
                labels=labels_erp,
                values=sizes_erp,
                marker=dict(colors=colors_erp),
                pull=[0, 0.1])
        ]
    )

    fig_erp.update_layout(
        title='Mercado ERP'
    )
    
    return fig.to_html(full_html=False, include_plotlyjs='cdn'), fig_erp.to_html(full_html=False, include_plotlyjs='cdn')


def create_interactive_map(df):
    """
    Generates a Bubble Map of companies.
    """
    df = df[['company_profile_cnae', 'province', 'GLOBAL_SCORE']].copy()
    df = df.groupby('province')['GLOBAL_SCORE'].mean().round(1).reset_index()

    def get_lat(row): return REGION_COORDINATES.get(row.get('province', 'Unknown'), REGION_COORDINATES['Unknown'])['lat']
    def get_lon(row): return REGION_COORDINATES.get(row.get('province', 'Unknown'), REGION_COORDINATES['Unknown'])['lon']
    
    df['lat_base'] = df.apply(get_lat, axis=1)
    df['lon_base'] = df.apply(get_lon, axis=1)

    fig = px.scatter_map(
        df,
        lat="lat_base",
        lon="lon_base",
        color="GLOBAL_SCORE",
        size=df['GLOBAL_SCORE'] * 0.3,
        color_continuous_scale=px.colors.sequential.Plasma,
        size_max=30,
        zoom=6,
        hover_name="province",
        hover_data={"lat_base": False, "lon_base": False, "GLOBAL_SCORE": True, "province": True},
        map_style="streets",
        title="Mapa de Madurez Digital (Geolocalizado)",
        labels={'GLOBAL_SCORE': 'Madurez Digital', 'province': 'Provincia'}
    )
    
    fig.update_layout(height=700, margin={"r":0,"t":40,"l":0,"b":0})
    
    return fig.to_html(full_html=False, include_plotlyjs='cdn')


def create_bubble_chart(df):
    df = df.copy()
    df['number_of_employees'] = pd.to_numeric(df['number_of_employees'], errors='coerce')
    df = df[df['number_of_employees'] > 0]

    def estimate_revenue(val):
        s = str(val)
        if '<1M' in s: return 500_000
        if '1-5M' in s: return 3_000_000
        if '5-20M' in s: return 12_500_000
        if '>20M' in s: return 50_000_000
        return np.nan

    df['est_revenue'] = df['annual_revenue'].apply(estimate_revenue)
    df['revenue_per_employee'] = df['est_revenue'] / df['number_of_employees']    
    plot_df = df.dropna(subset=['revenue_per_employee', 'GLOBAL_SCORE', 'number_of_employees'])

    if plot_df.empty:
        return "<div>No hay datos suficientes para calcular la eficiencia.</div>"

    # Create descriptive company size labels with employee ranges
    size_mapping = {
        'Micro': 'Micro (≤10 empleados)',
        'Pequeña': 'Pequeña (11-50 empleados)',
        'Mediana': 'Mediana (51-250 empleados)',
        'Grande': 'Grande (>250 empleados)'
    }
    plot_df = plot_df.copy()
    plot_df['company_size_desc'] = plot_df['company_size'].map(size_mapping)

    fig = px.scatter(
        plot_df,
        x='GLOBAL_SCORE',
        y='revenue_per_employee',
        size='number_of_employees',
        color='company_size_desc',
        
        hover_name='company_profile_cnae',
        hover_data={
            'company_size': False,
            'company_size_desc': False,
            'number_of_employees': ':.0f',
            'annual_revenue': True,
            'revenue_per_employee': ':.2s',
            'GLOBAL_SCORE': True
        },
        labels={
            'GLOBAL_SCORE': 'Índice de Madurez Digital (0-100)',
            'number_of_employees': 'Número de Empleados',
            'annual_revenue': 'Rango de Facturación Anual',
            'revenue_per_employee': 'Ingresos Estimados / Empleado (€)',
            'company_size_desc': 'Tamaño de Empresa'
        },
        
        color_discrete_map={
            'Micro (≤10 empleados)': '#2ecc71',
            'Pequeña (11-50 empleados)': '#3498db',
            'Mediana (51-250 empleados)': '#9b59b6',
            'Grande (>250 empleados)': '#e74c3c'
        }
    )

    fig.update_layout(
        height=600, 
        yaxis=dict(tickprefix="€"),
        legend=dict(orientation="h", y=-0.20)
    )
    
    return fig.to_html(full_html=False, include_plotlyjs='cdn')

def create_age_cloud_chart(df):
    fig = go.Figure(data=go.Heatmap(
        z=pd.crosstab(df['average_employee_age'], df['it_infrastructure_type']),
        x=['On-premise', 'Híbrida', 'Cloud'],
        y=["<30", "30-40", "41-50", ">50"],
        colorscale='Reds'
    ))

    fig.update_layout(
        xaxis_title='Tipo de Infraestructura TI',
        yaxis_title='Rango de Edad'
    )

    return fig.to_html(full_html=False, include_plotlyjs='cdn')


def create_digital_traction_chart(df):
    df = df.copy()

    def clean_digital_rev(val):
        s = str(val)
        if '<10%' in s: return 5
        if '10-30%' in s: return 20
        if '30-60%' in s: return 45
        if '>60%' in s: return 80
        if 'Nulo' in s or '0' in s: return 0
        return 0

    df['est_digital_revenue_pct'] = df['digital_revenue'].apply(clean_digital_rev)
    df['marketing_status'] = df['digital_marketing_use'].fillna('No')
    
    fig = px.box(
        df, 
        x='marketing_status', 
        y='est_digital_revenue_pct',
        color='marketing_status',
        points='all',
        
        hover_name='company_profile_cnae',
        hover_data={
            'annual_revenue': True,
            'digital_revenue': True,
            'est_digital_revenue_pct': False,
            'marketing_status': False
        },
        
        labels={
            'marketing_status': '¿Invierte en Marketing Digital?',
            'est_digital_revenue_pct': '% Estimado de Facturación Online'
        },
        
        color_discrete_map={
            'Sí': '#2ecc71',
            'No': '#95a5a6',
            'Parcial': '#3498db'
        }
    )

    fig.update_layout(
        height=400,
        yaxis=dict(ticksuffix="%", range=[-5, 100]),
        showlegend=False
    )
    
    return fig.to_html(full_html=False, include_plotlyjs='cdn')


def generate_dynamic_conclusion(df):
    avg_score = df['GLOBAL_SCORE'].mean()    
    dims = {
        'Operaciones': df['KPI_OPERATIONS'].mean(),
        'Ciberseguridad': df['KPI_SECURITY'].mean(),
        'Negocio Digital': df['KPI_BUSINESS'].mean(),
        'Cultura Digital': df['KPI_CULTURE'].mean()
    }

    strongest_dim = max(dims, key=dims.get)
    weakest_dim = min(dims, key=dims.get)
    
    if 'priority_assessment_area' in df.columns:
        top_priority = df['priority_assessment_area'].mode()[0]
    else:
        top_priority = "No especificado"

    
    if avg_score < 40:
        verdict_title = "Mercado en Etapa Inicial"
        verdict_text = f"El índice de madurez promedio ({avg_score:.1f}) indica que el sector se encuentra en una fase temprana de digitalización. La tecnología se usa de forma reactiva, no estratégica."
        color_class = "#e74c3c" # Red
    elif avg_score < 70:
        verdict_title = "Mercado en Desarrollo"
        verdict_text = f"Con un índice de {avg_score:.1f}, el mercado muestra avances sólidos, aunque dispares. Las empresas han adoptado herramientas básicas pero faltan procesos integrados."
        color_class = "#f39c12" # Orange
    else:
        verdict_title = "Mercado Maduro"
        verdict_text = f"El sector demuestra una alta competencia digital ({avg_score:.1f}). El desafío ya no es la adopción, sino la innovación y el uso de IA."
        color_class = "#2ecc71" # Green

    gap_text = f"El análisis revela que <strong>{weakest_dim}</strong> es el área crítica de mejora (puntuación: {dims[weakest_dim]:.1f}). Mientras que {strongest_dim} actúa como motor, el descuido en {weakest_dim} está frenando el crecimiento global."

    if top_priority in weakest_dim:
        strategy_text = f"Es positivo notar que las empresas son conscientes de su debilidad: la mayoría ha marcado <strong>{top_priority}</strong> como su prioridad, alineándose con los datos."
    else:
        strategy_text = f"Existe una <strong>disonancia estratégica</strong>: Aunque la mayor debilidad objetiva es {weakest_dim}, las empresas están priorizando invertir en <strong>{top_priority}</strong>. Se recomienda reevaluar este enfoque para cerrar brechas estructurales antes de buscar crecimiento."

    html_content = f"""
    <div style="background: white; padding: 25px; border-left: 6px solid {color_class}; border-radius: 5px; box-shadow: 0 2px 5px rgba(0,0,0,0.05);">
        <h3 style="color: {color_class}; margin-top: 0;">{verdict_title} (Promedio: {avg_score:.1f}/100)</h3>
        <p style="margin-bottom: 15px;">{verdict_text}</p>
        
        <h4 style="color: #0062a4; margin-bottom: 5px;">El factor limitante: {weakest_dim}</h4>
        <p style="margin-bottom: 15px;">{gap_text}</p>
        
        <h4 style="color: #0062a4; margin-bottom: 5px;">Recomendación Estratégica</h4>
        <p style="margin-bottom: 0;">{strategy_text}</p>
    </div>
    """
    
    return html_content

In [8]:
def generate_interactive_report(df):
    print(">>> Generating Interactive Report...")
    
    risk_pct = round((len(df[df['KPI_SECURITY'] < 40]) / len(df)) * 100, 1)
    methodology_table = create_kpi_methodology_table()
    scoring_logic = create_scoring_logic_html()
    chart_distplot_url = create_distplot_chart(df)

    jinja_data = {
        "date": datetime.date.today().strftime("%d %b %Y"),
        "total_companies": len(df),
        "avg_score": round(df['GLOBAL_SCORE'].mean(), 1),
        "risk_pct": risk_pct,
        "methodology_table": methodology_table,
        "scoring_logic": scoring_logic,
        "chart_radar_url": create_int_radar_chart(df),
        "chart_distplot_url": chart_distplot_url,
        "chart_boxplot_url": create_boxplot_chart(df),
        "chart_riskbar_url": create_riskbar_chart(df),
        "chart_priority_url": create_int_priority_chart(df),
        "chart_pie_erp_url": create_pie_chart(df)[0],
        "chart_pie_crm_url": create_pie_chart(df)[1],
        "chart_driver_erp_url": create_int_driver_charts(df)[0],
        "chart_driver_crm_url": create_int_driver_charts(df)[1],
        "chart_driver_cloud_url": create_int_driver_charts(df)[2],
        "chart_erp_distplot_url": create_erp_crm_distplot_chart(df)[0],
        "chart_crm_distplot_url": create_erp_crm_distplot_chart(df)[1],
        "chart_usage_crm_url": create_int_opportunies_charts(df)[0],
        "chart_usage_erp_url": create_int_opportunies_charts(df)[1],
        "chart_region_url": create_interactive_map(df),
        "chart_bubble_url": create_bubble_chart(df),
        "chart_age_cloud_url": create_age_cloud_chart(df),
        "chart_digital_traction_url": create_digital_traction_chart(df),
        "dynamic_conclusion_html": generate_dynamic_conclusion(df)
    }
        
    template = Template(INTERACTIVE_REPORT_TEMPLATE)
    
    with open("Informe_Consultoria_Interactivo.html", "w", encoding="utf-8") as f:
        f.write(template.render(jinja_data))
        
    print("[SUCCESS] Report 'Informe_Consultoria_Interactivo.html' created.")

generate_interactive_report(master_df)

>>> Generating Interactive Report...
[SUCCESS] Report 'Informe_Consultoria_Interactivo.html' created.
